In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from glob import glob
from sklearn.metrics import recall_score, f1_score, classification_report
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler 

In [ ]:
DATA_DIR = "/kaggle/input/extracted-fusion-embeddings/extracted_fusion_data"
BATCH_SIZE = 4                 
ACCUMULATE_STEPS = 8          
LEARNING_RATE = 1e-4
EPOCHS = 15
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 6
MAX_SEQ_LEN = 800              
NUM_CLASSES = 6
CLASS_NAMES = ["FP", "PW", "RP", "RV", "RS", "Fluent"]

In [ ]:
class DisfluencyDataset(Dataset):
    def __init__(self, file_paths):
        self.files = file_paths
        # MAPPING: Raw One-Hot Index (0-9) -> Target Class (0-5)
        self.label_map = {
            3: 0, 7: 1, 4: 2, 5: 3, 6: 4,  # Dis fluent
            1: 5, 0: 5, 2: 5, 8: 5, 9: 5   # Fluent
        }

    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
            path = self.files[idx]
            try:
                data = np.load(path)
                
                # Load data
                audio_np = data['audio_emb'] # (T, 768)
                text_np = data['text_emb']   # (T, 768)
                labels_onehot = data['labels'] # (T, 10)
                
                # If sequence is longer than MAX_SEQ_LEN, cut it.
                if audio_np.shape[0] > MAX_SEQ_LEN:
                    audio_np = audio_np[:MAX_SEQ_LEN]
                    text_np = text_np[:MAX_SEQ_LEN]
                    labels_onehot = labels_onehot[:MAX_SEQ_LEN]
                
                # Convert to Tensor
                audio = torch.from_numpy(audio_np).float()
                text = torch.from_numpy(text_np).float()
                
                # Label Decoding
                raw_indices = np.argmax(labels_onehot, axis=-1)
                mapped_labels = np.full_like(raw_indices, 5)
                for raw_id, target_id in self.label_map.items():
                    mapped_labels[raw_indices == raw_id] = target_id
                
                labels = torch.from_numpy(mapped_labels).long()
                return audio, text, labels
                
            except Exception as e:
                print(f"Error loading {path}: {e}")
                return None

In [ ]:
def get_split_files(data_dir):
    all_files = sorted(glob(os.path.join(data_dir, "*.npz")))
    train_f, dev_f, test_f = [], [], []
    for f_path in tqdm(all_files, desc="Partitioning"):
        fname = os.path.basename(f_path)
        if fname.startswith(("sw2", "sw3")): train_f.append(f_path)
        elif fname.startswith(("sw45", "sw46", "sw47", "sw48", "sw49")): dev_f.append(f_path)
        elif fname.startswith(("sw40", "sw41")): test_f.append(f_path)
    return train_f, dev_f, test_f

In [ ]:
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    audio_list, text_list, label_list = zip(*batch)
    
    audio_padded = nn.utils.rnn.pad_sequence(audio_list, batch_first=True, padding_value=0)
    text_padded = nn.utils.rnn.pad_sequence(text_list, batch_first=True, padding_value=0)
    labels_padded = nn.utils.rnn.pad_sequence(label_list, batch_first=True, padding_value=-100)
    
    lengths = torch.tensor([len(x) for x in audio_list])
    max_len = audio_padded.size(1)
    mask = torch.arange(max_len)[None, :] < lengths[:, None]
    
    return audio_padded, text_padded, labels_padded, mask

In [ ]:
class FusionModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512, num_heads=8, blstm_hidden=128, blstm_layers=1, num_classes=6, smoothing=True, smoothing_kernel=5, dropout=0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.smoothing = smoothing

        self.audio_proj = nn.Linear(input_dim, hidden_dim)
        self.text_proj = nn.Linear(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        self.attn_a2t = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)
        self.attn_t2a = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=num_heads, batch_first=True)
        self.norm_a = nn.LayerNorm(hidden_dim)
        self.norm_t = nn.LayerNorm(hidden_dim)

        self.gate_fc = nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.Sigmoid())
        self.fusion_norm = nn.LayerNorm(hidden_dim)

        self.blstm = nn.LSTM(input_size=hidden_dim, hidden_size=blstm_hidden, num_layers=blstm_layers, batch_first=True, bidirectional=True)
        
        self.classifier = nn.Sequential(
            nn.Linear(blstm_hidden * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

        if smoothing:
            pad = (smoothing_kernel - 1) // 2
            self.smooth_conv = nn.Conv1d(num_classes, num_classes, kernel_size=smoothing_kernel, padding=pad, groups=num_classes, bias=False)
            with torch.no_grad(): self.smooth_conv.weight.fill_(1.0 / smoothing_kernel)

    def forward(self, audio, text, mask_bool=None):
        a = self.dropout(F.gelu(self.audio_proj(audio)))
        t = self.dropout(F.gelu(self.text_proj(text)))

        kp_a = ~mask_bool if mask_bool is not None else None
        kp_t = ~mask_bool if mask_bool is not None else None

        a_enriched, _ = self.attn_a2t(query=a, key=t, value=t, key_padding_mask=kp_t)
        t_enriched, _ = self.attn_t2a(query=t, key=a, value=a, key_padding_mask=kp_a)

        a = self.norm_a(a + a_enriched)
        t = self.norm_t(t + t_enriched)

        concat = torch.cat([a, t], dim=-1)
        gate = self.gate_fc(concat)
        fused = gate * a + (1.0 - gate) * t
        fused = self.fusion_norm(fused)

        if mask_bool is not None: fused = fused * mask_bool.unsqueeze(-1).float()

        blstm_out, _ = self.blstm(fused)
        logits = self.classifier(blstm_out)

        if self.smoothing:
            logits = logits.permute(0, 2, 1)
            logits = self.smooth_conv(logits)
            logits = logits.permute(0, 2, 1)

        return logits

In [ ]:
def main():
    torch.cuda.empty_cache() # Clear any leftover RAM
    
    train_files, dev_files, _ = get_split_files(DATA_DIR)
    train_ds = DisfluencyDataset(train_files)
    dev_ds = DisfluencyDataset(dev_files)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
    
    print(f"Initializing FusionModel on {DEVICE}...")
    model = FusionModel(num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scaler = GradScaler() # AMP Scaler
    
    class_weights = torch.tensor([5.0, 5.0, 5.0, 5.0, 5.0, 1.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
    
    best_uar = 0.0
    print("\n--- Starting Training (AMP Enabled) ---")
    
    for epoch in range(EPOCHS):
        # --- TRAIN ---
        model.train()
        total_loss = 0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for i, batch in enumerate(pbar):
            if batch is None: continue
            audio, text, labels, mask = [x.to(DEVICE) for x in batch]
            
            # Mixed Precision Training
            with autocast():
                logits = model(audio, text, mask_bool=mask) 
                loss = criterion(logits.reshape(-1, NUM_CLASSES), labels.reshape(-1))
                loss = loss / ACCUMULATE_STEPS 

            scaler.scale(loss).backward()
            
            if (i + 1) % ACCUMULATE_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            total_loss += loss.item() * ACCUMULATE_STEPS
            pbar.set_postfix({'loss': f"{loss.item() * ACCUMULATE_STEPS:.4f}"})
            
        # --- VALIDATE ---
        model.eval()
        all_preds, all_labels = [], []
        print("Validating...")
        with torch.no_grad():
            for batch in dev_loader:
                if batch is None: continue
                audio, text, labels, mask = [x.to(DEVICE) for x in batch]
                
                with autocast():
                    logits = model(audio, text, mask_bool=mask)
                
                preds = torch.argmax(logits, dim=-1)
                
                # Mask out padding (-100)
                mask_flat = labels != -100
                all_preds.extend(preds[mask_flat].cpu().numpy())
                all_labels.extend(labels[mask_flat].cpu().numpy())
        
        val_uar = recall_score(all_labels, all_preds, average='macro')
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Loss: {total_loss/len(train_loader):.4f}")
        print(f"  UAR:  {val_uar:.4f} (Primary Metric)")
        print(f"  F1:   {val_f1:.4f}")
        
        # The Full Report Table
        print("\nDetailed Classification Report:")
        print(classification_report(
            all_labels, 
            all_preds, 
            labels=list(range(NUM_CLASSES)),  
            target_names=CLASS_NAMES, 
            digits=4, 
            zero_division=0
        ))
        
        if val_uar > best_uar:
            best_uar = val_uar
            torch.save(model.state_dict(), "best_fusion_model.pth")
            print("New Best Model Saved!")
        print("="*60)

        # 2. Save Last Model (Always overwrites at end of epoch)
        torch.save(model.state_dict(), "last_fusion_model.pth")
        print("Last Model Saved")
    

if __name__ == "__main__":
    main()